In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import mean_absolute_error, mean_squared_error
import pickle
from sklearn.metrics import r2_score



In [ ]:


with open('../models/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

In [ ]:
train = pd.read_csv('../data/processed/train_engineered.csv')
test = pd.read_csv('../data/processed/test_engineered.csv')

In [ ]:
cols_to_scale=['units_sold',
'unit_price',
'stock_on_hand',
'reorder_point',
'discount_pct',
'supplier_lead_days',
'lag_7', 'lag_14',
'rolling_mean_7', 'rolling_std_7',
'rolling_mean_30', 'rolling_std_30']

In [ ]:
feature_cols = ['unit_price','stock_on_hand','reorder_point','is_promotion','discount_pct','day_of_week', 'month','supplier_lead_days','Beauty','Electronics','Home','Sports','lag_7','lag_14','rolling_mean_7','rolling_std_7','rolling_mean_30','rolling_std_30','is_weekend','quarter','quarter_sin','quarter_cos']

X_train = train[feature_cols]
y_train = train['units_sold']

X_test = test[feature_cols]
y_test = test['units_sold']


In [ ]:
class InventoryDataset(Dataset):
    def __init__(self, X, y,window_size=14):
        self.X = X
        self.y = y
        self.window_size = window_size

    def __len__(self):
        return len(self.X) - self.window_size

    def __getitem__(self, idx):
        x = self.X.iloc[idx:idx+self.window_size].values
        y = self.y.iloc[idx+self.window_size]
            

       

        return torch.FloatTensor(x), torch.FloatTensor([y])

        

In [ ]:
train_dataset = InventoryDataset(X_train, y_train)
test_dataset = InventoryDataset(X_test, y_test)

In [ ]:


train_loader = DataLoader(train_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self):
        super(LSTMModel, self).__init__()

        self.lstm = nn.LSTM(
            input_size=22,
            hidden_size=128,
            num_layers=2,
            batch_first=True
        )

        
        self.fc = nn.Linear(128, 1)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)

        last_out = lstm_out[:, -1, :]


        out = self.fc(last_out)

        return out

In [ ]:
model = LSTMModel()

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

In [ ]:
epochs = 150

for epoch in range(epochs):

    model.train()

    epoch_loss = 0

    for X_batch, y_batch in train_loader:

        # Move data to GPU/CPU
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # Forward pass
        predictions = model(X_batch)

        # Calculate loss
        loss = criterion(predictions, y_batch)

        # Clear old gradients
        optimizer.zero_grad()

        # Backpropagation
        loss.backward()
    
        # Update weights
        optimizer.step()

        # Store batch loss
        epoch_loss += loss.item()

    # Average loss for entire epoch
    avg_loss = epoch_loss / len(train_loader)

    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

Epoch 1/100, Loss: 0.0080
Epoch 2/100, Loss: 0.0053
Epoch 3/100, Loss: 0.0045
Epoch 4/100, Loss: 0.0043
Epoch 5/100, Loss: 0.0043
Epoch 6/100, Loss: 0.0042
Epoch 7/100, Loss: 0.0041
Epoch 8/100, Loss: 0.0041
Epoch 9/100, Loss: 0.0040
Epoch 10/100, Loss: 0.0040
Epoch 11/100, Loss: 0.0040
Epoch 12/100, Loss: 0.0039
Epoch 13/100, Loss: 0.0039
Epoch 14/100, Loss: 0.0039
Epoch 15/100, Loss: 0.0039
Epoch 16/100, Loss: 0.0038
Epoch 17/100, Loss: 0.0038
Epoch 18/100, Loss: 0.0038
Epoch 19/100, Loss: 0.0038
Epoch 20/100, Loss: 0.0038
Epoch 21/100, Loss: 0.0038
Epoch 22/100, Loss: 0.0038
Epoch 23/100, Loss: 0.0038
Epoch 24/100, Loss: 0.0037
Epoch 25/100, Loss: 0.0037
Epoch 26/100, Loss: 0.0037
Epoch 27/100, Loss: 0.0037
Epoch 28/100, Loss: 0.0037
Epoch 29/100, Loss: 0.0037
Epoch 30/100, Loss: 0.0037
Epoch 31/100, Loss: 0.0037
Epoch 32/100, Loss: 0.0037
Epoch 33/100, Loss: 0.0037
Epoch 34/100, Loss: 0.0037
Epoch 35/100, Loss: 0.0037
Epoch 36/100, Loss: 0.0037
Epoch 37/100, Loss: 0.0037
Epoch 38/1

In [ ]:
model.eval()

test_loss = 0

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        # Move data to GPU/CPU
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # Predictions
        predictions = model(X_batch)

        # Calculate loss
        loss = criterion(predictions, y_batch)

        # Add batch loss
        test_loss += loss.item()

# Average test loss
avg_test_loss = test_loss / len(test_loader)

print(f"Test Loss: {avg_test_loss:.4f}")

Test Loss: 0.0046


In [ ]:
torch.save(model.state_dict(), '../models/lstm_model.pt')

print("Model Saved Successfully!")

Model Saved Successfully!


In [ ]:
class MLPModel(nn.Module):

    def __init__(self):

        super(MLPModel, self).__init__()

        self.fc1 = nn.Linear(22, 128)
        self.relu1 = nn.ReLU()

        self.fc2 = nn.Linear(128, 64)
        self.relu2 = nn.ReLU()

        self.fc3 = nn.Linear(64, 1)

    def forward(self, x):

        x = self.fc1(x)
        x = self.relu1(x)

        x = self.fc2(x)
        x = self.relu2(x)

        x = self.fc3(x)

        return x

In [ ]:
class MLPDataset(Dataset):
    def __init__(self, X, y):
        self.X = X.values
        self.y = y.values

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return (
            torch.FloatTensor(self.X[idx]),
            torch.FloatTensor([self.y[idx]])
        )

In [ ]:
lstm_test_dataset = InventoryDataset(X_test, y_test)
lstm_test_loader = DataLoader(lstm_test_dataset, batch_size=64, shuffle=False)

In [ ]:
mlp_train_dataset = MLPDataset(X_train, y_train)
mlp_test_dataset = MLPDataset(X_test, y_test)
mlp_test_loader = DataLoader(mlp_test_dataset, batch_size=64, shuffle=False)
mlp_loader = DataLoader(mlp_train_dataset, batch_size=64, shuffle=False)

mlp_model = MLPModel().to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(mlp_model.parameters(), lr=0.001)

In [ ]:
epochs = 150

for epoch in range(epochs):

    mlp_model.train()
    epoch_loss = 0

    for X_batch, y_batch in mlp_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        predictions = mlp_model(X_batch)

        loss = criterion(predictions, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(mlp_loader)

    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

Epoch 1/100, Loss: 0.0079
Epoch 2/100, Loss: 0.0027
Epoch 3/100, Loss: 0.0021
Epoch 4/100, Loss: 0.0019
Epoch 5/100, Loss: 0.0018
Epoch 6/100, Loss: 0.0018
Epoch 7/100, Loss: 0.0017
Epoch 8/100, Loss: 0.0018
Epoch 9/100, Loss: 0.0018
Epoch 10/100, Loss: 0.0020
Epoch 11/100, Loss: 0.0023
Epoch 12/100, Loss: 0.0016
Epoch 13/100, Loss: 0.0016
Epoch 14/100, Loss: 0.0015
Epoch 15/100, Loss: 0.0015
Epoch 16/100, Loss: 0.0015
Epoch 17/100, Loss: 0.0015
Epoch 18/100, Loss: 0.0015
Epoch 19/100, Loss: 0.0015
Epoch 20/100, Loss: 0.0017
Epoch 21/100, Loss: 0.0018
Epoch 22/100, Loss: 0.0016
Epoch 23/100, Loss: 0.0014
Epoch 24/100, Loss: 0.0014
Epoch 25/100, Loss: 0.0014
Epoch 26/100, Loss: 0.0014
Epoch 27/100, Loss: 0.0014
Epoch 28/100, Loss: 0.0014
Epoch 29/100, Loss: 0.0014
Epoch 30/100, Loss: 0.0015
Epoch 31/100, Loss: 0.0015
Epoch 32/100, Loss: 0.0014
Epoch 33/100, Loss: 0.0014
Epoch 34/100, Loss: 0.0013
Epoch 35/100, Loss: 0.0014
Epoch 36/100, Loss: 0.0014
Epoch 37/100, Loss: 0.0014
Epoch 38/1

In [ ]:
def evaluate(model, loader, is_lstm=False):

    model.eval()

    y_true, y_pred = [], []

    with torch.no_grad():

        for X_batch, y_batch in loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            if is_lstm:
                preds = model(X_batch)
            else:
                if len(X_batch.shape) == 3:
                    X_batch = X_batch[:, -1, :]
                preds = model(X_batch)

            y_true.extend(y_batch.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    y_true = np.array(y_true).flatten()
    y_pred = np.array(y_pred).flatten()

    n_features = len(cols_to_scale)
    target_idx = cols_to_scale.index("units_sold")

    dummy_true = np.zeros((len(y_true), n_features))
    dummy_pred = np.zeros((len(y_pred), n_features))

    dummy_true[:, target_idx] = y_true
    dummy_pred[:, target_idx] = y_pred


    y_true_real = scaler.inverse_transform(dummy_true)[:, target_idx]
    y_pred_real = scaler.inverse_transform(dummy_pred)[:, target_idx]

    

    mae = mean_absolute_error(y_true_real, y_pred_real)
    rmse = np.sqrt(mean_squared_error(y_true_real, y_pred_real))
    mape = np.mean(np.abs((y_true_real - y_pred_real) / (y_true_real + 1e-8))) * 100
    r2 = r2_score(y_true_real, y_pred_real)

    return mae, rmse, mape, r2

In [ ]:
lstm_mae, lstm_rmse, lstm_mape, lstm_r2 = evaluate(model, lstm_test_loader, is_lstm=True)

mlp_mae, mlp_rmse, mlp_mape, mlp_r2 = evaluate(mlp_model, mlp_test_loader, is_lstm=False)

In [ ]:
results = pd.DataFrame({
    "Model": ["LSTM", "MLP"],
    "MAE": [lstm_mae, mlp_mae],
    "RMSE": [lstm_rmse, mlp_rmse],
    "MAPE": [lstm_mape, mlp_mape],
    "R2": [lstm_r2, mlp_r2]
})

print(results)

  Model       MAE       RMSE       MAPE        R2
0  LSTM  6.974557  10.291286  25.183894  0.506163
1   MLP  4.494347   5.790845  17.539544  0.841754


In [ ]:
print(test['Beauty'].value_counts())
print(test.columns.tolist())